# Population Genetics and Molecular Evolution

**Tier 3 -- Applied Bioinformatics**

Population genetics explains how allele frequencies shift across generations under the competing forces of mutation, genetic drift, natural selection, and gene flow. Molecular evolution extends these ideas to DNA sequences, asking which substitutions are neutral, purifying, or adaptive.

**Learning objectives:**
- Compute allele frequencies and test Hardy-Weinberg equilibrium with chi-squared
- Simulate Wright-Fisher genetic drift and understand the effect of population size
- Model directional, balancing, purifying, and frequency-dependent selection
- Apply the molecular clock and Jukes-Cantor correction to estimate divergence times
- Calculate dN/dS ratios with the Nei-Gojobori method and interpret selection pressure
- Compute Tajima's D, McDonald-Kreitman neutrality index, Fst, and linkage disequilibrium

**Prerequisites:** Tier 2 Python, NumPy, basic probability  
**Libraries:** `numpy`, `pandas`, `matplotlib`, `scipy`

---[< Previous: Population Genetics + Molecular Evolution: Over...](01_population_genetics_and_molecular_evolution.ipynb) | [Tier 3: Applied Bioinformatics Overview](../README.md) | [Next: Numerical Methods for Bioinformatics >](../16_Numerical_Methods_for_Bioinformatics/01_numerical_methods_for_bioinformatics.ipynb)---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from itertools import combinations

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12
np.random.seed(42)

---

## 1. Allele Frequencies and Hardy-Weinberg Equilibrium

The **Hardy-Weinberg principle** (1908) states that, in an infinitely large, randomly mating population with no mutation, selection, or migration, allele frequencies remain constant across generations and genotype frequencies satisfy:

$$p^2 + 2pq + q^2 = 1$$

where *p* = frequency of allele A and *q* = 1 − *p* = frequency of allele a.

HWE is a null model: departures signal interesting biology.

In [ ]:
# Observed genotype counts from a sample of 500 individuals
# Locus: MN blood group (codominant: MM, MN, NN)
observed_MM = 233
observed_MN = 200
observed_NN = 67
n_individuals = observed_MM + observed_MN + observed_NN

# Allele frequencies
n_M_alleles = 2 * observed_MM + observed_MN
n_N_alleles = 2 * observed_NN + observed_MN
total_alleles = 2 * n_individuals

p_M = n_M_alleles / total_alleles   # frequency of M
q_N = n_N_alleles / total_alleles   # frequency of N

print(f"Sample size (n): {n_individuals} individuals")
print(f"Allele counts  -- M: {n_M_alleles}, N: {n_N_alleles}")
print(f"Allele frequencies -- p(M) = {p_M:.4f}, q(N) = {q_N:.4f}")
print(f"p + q = {p_M + q_N:.4f}  (must equal 1)")

In [ ]:
# HWE expected genotype frequencies and bar chart comparison
expected_MM = p_M**2 * n_individuals
expected_MN = 2 * p_M * q_N * n_individuals
expected_NN = q_N**2 * n_individuals

genotypes  = ['MM', 'MN', 'NN']
obs_counts = [observed_MM, observed_MN, observed_NN]
exp_counts = [expected_MM, expected_MN, expected_NN]

x = np.arange(len(genotypes))
width = 0.35
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - width/2, obs_counts, width, label='Observed', color='steelblue')
ax.bar(x + width/2, exp_counts, width, label='HWE Expected', color='coral', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(genotypes)
ax.set_ylabel('Count')
ax.set_title('MN Blood Group: Observed vs HWE Expected')
ax.legend()
plt.tight_layout()
plt.show()

print(f"  MM: observed={observed_MM}, expected={expected_MM:.1f}")
print(f"  MN: observed={observed_MN}, expected={expected_MN:.1f}")
print(f"  NN: observed={observed_NN}, expected={expected_NN:.1f}")

In [ ]:
# Chi-squared test for HWE departure
# df = (number of genotype classes - 1) - (independent allele freqs estimated)
# For biallelic locus: df = 3 - 1 - 1 = 1
observed_arr = np.array([observed_MM, observed_MN, observed_NN], dtype=float)
expected_arr = np.array([expected_MM, expected_MN, expected_NN], dtype=float)

chi2_stat = np.sum((observed_arr - expected_arr)**2 / expected_arr)
df = 1
p_value = stats.chi2.sf(chi2_stat, df)   # survival function = 1 - CDF

print(f"Chi-squared statistic: {chi2_stat:.4f}")
print(f"Degrees of freedom:    {df}")
print(f"p-value:               {p_value:.4f}")
print()
if p_value > 0.05:
    print("Conclusion: No significant departure from HWE (p > 0.05).")
else:
    print("Conclusion: Significant departure from HWE (p <= 0.05).")

### Causes of HWE Departure

| Force | Effect on allele freq | Effect on genotype freq |
|---|---|---|
| Natural selection | Changes p and q | Excess/deficit of certain genotypes |
| Genetic drift | Changes p and q (random) | Excess homozygotes (inbreeding-like) |
| Gene flow (migration) | Introduces new alleles | Shifts all genotype freqs |
| Mutation | Slow, negligible per generation | Tiny shift toward new alleles |
| Non-random mating | No direct effect on p, q | Excess homozygotes (inbreeding) or excess heterozygotes |
| Population stratification | No effect within strata | Wahlund effect: excess homozygotes in pooled sample |

The **inbreeding coefficient F** measures the fractional reduction in heterozygosity:
$$F = 1 - \frac{H_{\text{obs}}}{2pq}$$

In [ ]:
# Inbreeding coefficient F and multi-allelic HWE extension
H_obs = observed_MN / n_individuals
H_exp = 2 * p_M * q_N
F_inbreeding = 1 - H_obs / H_exp

print(f"Inbreeding coefficient F = {F_inbreeding:.4f}")
print(f"  H_obs={H_obs:.4f}, H_exp={H_exp:.4f}")
print("  F ≈ 0: no evidence of inbreeding or outbreeding.")
print()

# Multi-allelic HWE for a 3-allele ABO-like locus
p_A, p_B, p_O = 0.28, 0.06, 0.66
allele_freqs = {'A': p_A, 'B': p_B, 'O': p_O}
alleles = list(allele_freqs.keys())

print("Expected genotype frequencies under HWE (3-allele ABO-like locus):")
total = 0.0
for i, a1 in enumerate(alleles):
    for a2 in alleles[i:]:
        freq = allele_freqs[a1]**2 if a1 == a2 else 2*allele_freqs[a1]*allele_freqs[a2]
        total += freq
        print(f"  {a1}{a2}: {freq:.4f}")
print(f"  Sum: {total:.4f}")

---

## 2. Genetic Drift

**Genetic drift** is the random change in allele frequency caused by sampling a finite number of gametes each generation. It is the dominant evolutionary force in small populations.

### Wright-Fisher Model

In a diploid population of size *N*, the next generation's count of allele A is drawn from a binomial distribution:

$$k_{t+1} \sim \text{Binomial}(2N,\; p_t)$$

Variance per generation: $\text{Var}(p_{t+1}) = p_t(1-p_t)/(2N)$

The allele frequency performs a random walk bounded by absorbing states at 0 (loss) and 1 (fixation). Smaller N means larger variance and faster drift.

### Absorbing States: Fixation and Loss

Once an allele reaches frequency 0 or 1, it stays there forever (in the absence of mutation).
These are called **absorbing states** of the Wright-Fisher Markov chain.

- **Fixation** (p = 1): the allele has replaced all others — it is now the only allele at this locus
- **Loss** (p = 0): the allele has been eliminated from the population

For a neutral allele starting at frequency p₀ in a population of size N:
- P(fixation) = p₀
- P(loss) = 1 − p₀
- Expected time to fixation (given fixation) ≈ −4N × [p₀ ln(p₀) + (1-p₀) ln(1-p₀)] / p₀


In [ ]:
def wright_fisher_trajectory(N, p0, n_generations, rng):
    """Simulate one Wright-Fisher trajectory.
    Returns array of allele frequencies, length n_generations+1."""
    freq = np.empty(n_generations + 1)
    freq[0] = p0
    p = p0
    for g in range(n_generations):
        k = rng.binomial(2 * N, p)
        p = k / (2 * N)
        freq[g + 1] = p
        if p == 0.0 or p == 1.0:
            freq[g + 2:] = p   # absorbing state: stay at 0 or 1
            break
    return freq

rng = np.random.default_rng(42)

# Single trajectory for three different population sizes
n_gen = 200
p_initial = 0.5
pop_sizes = [20, 100, 1000]
colors = ['firebrick', 'steelblue', 'seagreen']

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, N, color in zip(axes, pop_sizes, colors):
    traj = wright_fisher_trajectory(N, p_initial, n_gen, rng)
    ax.plot(traj, color=color, lw=1.5)
    ax.axhline(0.0, ls='--', color='black', lw=0.8)
    ax.axhline(1.0, ls='--', color='black', lw=0.8)
    ax.set_title(f'N = {N}')
    ax.set_xlabel('Generation')
    ax.set_ylim(-0.05, 1.05)
axes[0].set_ylabel('Allele frequency (p)')
fig.suptitle('Wright-Fisher Drift: Single Trajectory per Population Size')
plt.tight_layout()
plt.show()

In [ ]:
# Many replicates: visualise spread, fixation, and loss
# Under neutral drift, P(fixation) = p0 (Kimura 1962)
n_replicates = 50
n_gen = 150
N_small = 50
p0 = 0.3

trajectories = np.array([
    wright_fisher_trajectory(N_small, p0, n_gen, rng)
    for _ in range(n_replicates)
])

n_fixed = np.sum(trajectories[:, -1] == 1.0)
n_lost  = np.sum(trajectories[:, -1] == 0.0)
n_poly  = n_replicates - n_fixed - n_lost

fig, ax = plt.subplots(figsize=(12, 5))
for traj in trajectories:
    final = traj[-1]
    col = 'steelblue' if final == 1.0 else ('tomato' if final == 0.0 else 'lightgray')
    ax.plot(traj, color=col, alpha=0.6, lw=0.9)

ax.axhline(p0, ls=':', color='black', lw=1.2, label=f'Initial p = {p0}')
ax.set_xlabel('Generation')
ax.set_ylabel('Allele frequency')
ax.set_title(f'N={N_small}, p0={p0}: {n_replicates} replicates | '
             f'Fixed={n_fixed} (blue), Lost={n_lost} (red), Polymorphic={n_poly} (gray)')
ax.legend()
plt.tight_layout()
plt.show()
print(f"Theoretical P(fixation) = p0 = {p0:.2f}")
print(f"Observed fixation fraction   = {n_fixed/n_replicates:.2f}")

In [ ]:
# Time to fixation or loss as a function of N
# Theory: E[T_fix or loss] ≈ -4N[p*ln(p) + (1-p)*ln(1-p)]

def mean_time_to_fixation_loss(N, p0, n_replicates, rng, max_gen=10_000):
    times = []
    for _ in range(n_replicates):
        p = p0
        for g in range(1, max_gen + 1):
            p = rng.binomial(2 * N, p) / (2 * N)
            if p == 0.0 or p == 1.0:
                times.append(g)
                break
    return np.mean(times) if times else np.nan

pop_sizes_test = [10, 25, 50, 100, 200]
mean_times = [mean_time_to_fixation_loss(N, 0.5, 200, rng) for N in pop_sizes_test]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(pop_sizes_test, mean_times, 'o-', color='steelblue', lw=2, label='Simulated')
theory_x = np.linspace(10, 200, 100)
theory_y = -4 * theory_x * (0.5 * np.log(0.5) + 0.5 * np.log(0.5))
ax.plot(theory_x, theory_y, '--', color='coral', label='Theory: −4N[p ln p + q ln q]')
ax.set_xlabel('Population size (N)')
ax.set_ylabel('Mean generations to fixation/loss')
ax.set_title('Time to Fixation or Loss vs Population Size (p₀ = 0.5)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Bottleneck: simulating a transient reduction in Ne

def wf_with_bottleneck(N_normal, N_bottleneck, bottleneck_start, bottleneck_duration,
                        p0, n_gen, rng):
    """Wright-Fisher drift with a population size bottleneck."""
    freq = np.empty(n_gen + 1)
    freq[0] = p0
    p = p0
    for g in range(n_gen):
        in_bottleneck = bottleneck_start <= g < bottleneck_start + bottleneck_duration
        N = N_bottleneck if in_bottleneck else N_normal
        p = rng.binomial(2 * N, p) / (2 * N)
        freq[g + 1] = p
        if p == 0.0 or p == 1.0:
            freq[g + 2:] = p
            break
    return freq

n_gen = 200
n_reps = 30
bottle_trajs = [wf_with_bottleneck(500, 5, 50, 10, 0.5, n_gen, rng) for _ in range(n_reps)]
normal_trajs = [wright_fisher_trajectory(500, 0.5, n_gen, rng) for _ in range(n_reps)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for traj in normal_trajs:
    ax1.plot(traj, color='steelblue', alpha=0.4, lw=0.8)
ax1.set_title('No bottleneck (N=500 throughout)')

for traj in bottle_trajs:
    ax2.plot(traj, color='tomato', alpha=0.4, lw=0.8)
ax2.axvspan(50, 60, color='orange', alpha=0.2, label='Bottleneck (N=5, 10 gen)')
ax2.set_title('Bottleneck at generation 50-60 (N drops to 5)')
ax2.legend()

for ax in (ax1, ax2):
    ax.set_xlabel('Generation')
    ax.set_ylabel('Allele frequency')
plt.suptitle('Bottleneck Effects on Drift', y=1.01)
plt.tight_layout()
plt.show()

### Effective Population Size (Ne)

The **effective population size (Ne)** is the size of an idealized Wright-Fisher population that experiences the same rate of genetic drift as the real population. Ne is almost always smaller than the census size N due to:

- Unequal sex ratios: $N_e = 4N_mN_f/(N_m + N_f)$
- Variance in reproductive success (high-variance breeders reduce Ne)
- Population size fluctuations: Ne ≈ harmonic mean of per-generation sizes
- Overlapping generations

Human Ne ≈ 10,000 despite a census size of ~8 billion, primarily because of historical bottlenecks and recent exponential growth.

In [ ]:
# Harmonic mean Ne across fluctuating population sizes
census_sizes = np.array([10_000, 10_000, 500, 10_000, 10_000, 200, 10_000, 10_000])
Ne_harmonic = len(census_sizes) / np.sum(1.0 / census_sizes)

print("Census sizes across 8 generations:", census_sizes)
print(f"Arithmetic mean census size: {census_sizes.mean():.0f}")
print(f"Harmonic mean (= Ne):        {Ne_harmonic:.0f}")
print(f"Ratio Ne/N_arithmetic:       {Ne_harmonic/census_sizes.mean():.3f}")
print()
print("Two brief crashes to N=500 and N=200 reduce Ne by >90%")
print("relative to the arithmetic mean.")

# Sex ratio effect
N_m, N_f = 100, 1000   # 10x more breeding females than males
Ne_sex = 4 * N_m * N_f / (N_m + N_f)
print(f"\nSex ratio example: N_m={N_m}, N_f={N_f}")
print(f"  Census N = {N_m + N_f}, Ne(sex ratio) = {Ne_sex:.0f}")

In [ ]:
# Drift dispersion: variance in allele frequency grows over time
# Var(p_t) = p0*(1-p0) * [1 - (1 - 1/(2N))^t]
# Asymptotically approaches p0*(1-p0) (all loci fixed or lost)

N_disp = 200
p0_disp = 0.5
n_gen_disp = 500
n_reps_disp = 100
rng_disp = np.random.default_rng(55)

traj_matrix = np.array([
    wright_fisher_trajectory(N_disp, p0_disp, n_gen_disp, rng_disp)
    for _ in range(n_reps_disp)
])

gen_range = np.arange(n_gen_disp + 1)
var_simulated = np.var(traj_matrix, axis=0)
var_theory = p0_disp * (1 - p0_disp) * (1 - (1 - 1/(2*N_disp))**gen_range)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(gen_range, var_simulated, color='steelblue', lw=1.5, label='Simulated variance')
ax.plot(gen_range, var_theory, '--', color='coral', lw=2, label='Theory: p₀(1-p₀)[1-(1-1/2N)^t]')
ax.set_xlabel('Generation')
ax.set_ylabel('Variance in allele frequency')
ax.set_title(f'Drift Dispersion Over Time (N={N_disp}, p₀={p0_disp}, {n_reps_disp} reps)')
ax.legend()
plt.tight_layout()
plt.show()


---

## 3. Natural Selection

Selection changes allele frequencies deterministically. The key parameters are:

| Parameter | Symbol | Meaning |
|---|---|---|
| Selection coefficient | *s* | Fitness disadvantage of *aa* relative to *AA* |
| Dominance coefficient | *h* | Fitness of heterozygote (*Aa*) relative to homozygotes |
| Fitness of *AA* | $w_{AA}$ | 1 |
| Fitness of *Aa* | $w_{Aa}$ | 1 − *hs* |
| Fitness of *aa* | $w_{aa}$ | 1 − *s* |

Mean fitness: $\bar{w} = p^2 + 2pq(1-hs) + q^2(1-s)$

Allele frequency after one generation:
$$p' = \frac{p[p \cdot w_{AA} + q \cdot w_{Aa}]}{\bar{w}}$$

**Key threshold**: selection dominates drift when $2N_es \gg 1$; drift dominates when $2N_es \ll 1$.

In [ ]:
def selection_trajectory(p0, s, h, n_gen):
    """Deterministic selection trajectory. A is advantageous over a.
    aa: fitness 1-s, Aa: fitness 1-h*s, AA: fitness 1.
    """
    p = p0
    freqs = [p]
    for _ in range(n_gen):
        q = 1 - p
        w_bar = p**2 + 2*p*q*(1-h*s) + q**2*(1-s)
        p = (p**2 + p*q*(1-h*s)) / w_bar
        freqs.append(p)
        if p > 0.9999:
            freqs.extend([1.0] * (n_gen - len(freqs) + 1))
            break
    return np.array(freqs[:n_gen + 1])

# Directional selection: varying s, additive dominance (h=0.5)
n_gen = 400
p0 = 0.01
s_values = [0.01, 0.05, 0.10, 0.20]
colors = ['navy', 'steelblue', 'coral', 'firebrick']

fig, ax = plt.subplots(figsize=(10, 5))
for s, color in zip(s_values, colors):
    traj = selection_trajectory(p0, s, h=0.5, n_gen=n_gen)
    ax.plot(traj, color=color, lw=2, label=f's = {s}')
ax.set_xlabel('Generation')
ax.set_ylabel('Frequency of advantageous allele (p)')
ax.set_title('Directional Selection: Selective Sweeps (h=0.5, p₀=0.01)')
ax.legend(title='Selection coefficient')
plt.tight_layout()
plt.show()

In [ ]:
# Balancing selection: heterozygote advantage (overdominance)
# Sickle-cell anemia: HbA/HbS heterozygotes most fit in malaria-endemic regions

def overdominance_trajectory(p0, s_AA, s_SS, n_gen):
    """p = frequency of HbS allele. Heterozygote (HbA/HbS) has fitness 1."""
    p = p0
    freqs = [p]
    w_hom_A = 1.0 - s_AA   # fully susceptible to malaria
    w_het   = 1.0           # heterozygote (fittest)
    w_hom_S = 1.0 - s_SS   # sickle-cell disease
    for _ in range(n_gen):
        q = 1 - p
        w_bar = p**2*w_hom_A + 2*p*q*w_het + q**2*w_hom_S
        p = (p**2*w_hom_A + p*q*w_het) / w_bar
        freqs.append(p)
    return np.array(freqs)

s_malaria = 0.10
s_sickle  = 0.90
p_eq = s_malaria / (s_malaria + s_sickle)
print(f"Predicted equilibrium frequency of HbS: {p_eq:.3f}")

fig, ax = plt.subplots(figsize=(10, 5))
for p_start, color in [(0.01, 'steelblue'), (0.5, 'coral'), (0.99, 'seagreen')]:
    traj = overdominance_trajectory(p_start, s_malaria, s_sickle, n_gen=150)
    ax.plot(traj, color=color, lw=2, label=f'p₀ = {p_start}')
ax.axhline(p_eq, ls='--', color='black', lw=1.5, label=f'Equilibrium p* = {p_eq:.3f}')
ax.set_xlabel('Generation')
ax.set_ylabel('Frequency of HbS allele')
ax.set_title('Balancing Selection: Sickle-Cell Heterozygote Advantage')
ax.legend()
plt.tight_layout()
plt.show()

### Frequency-Dependent Selection

In **negative frequency-dependent selection**, the fitness of an allele decreases as it becomes more common. This is common in:
- Immune escape: pathogens rare surface variants evade host memory
- Plant self-incompatibility: rare alleles have more compatible mates
- MHC diversity: rare alleles recognize new pathogens

A simple model: $w_A = 1 - s(p - \hat{p})$ and $w_a = 1 - s(q - \hat{q})$ where $\hat{p}$ is the equilibrium frequency. This always stabilizes allele frequency at $\hat{p}$.

In [ ]:
# Frequency-dependent selection: negative frequency dependence
# Fitness of A decreases linearly as A becomes more common
# w_A(p) = 1 + s*(0.5 - p),  w_a(p) = 1 + s*(0.5 - (1-p))
# Equilibrium at p = 0.5

def freq_dep_selection_trajectory(p0, s_freq_dep, n_gen):
    """Negative frequency-dependent selection. Equilibrium at p=0.5."""
    p = p0
    freqs = [p]
    for _ in range(n_gen):
        q = 1 - p
        w_A = 1 + s_freq_dep * (0.5 - p)   # lower fitness when common
        w_a = 1 + s_freq_dep * (0.5 - q)   # lower fitness when common
        w_bar = p * w_A + q * w_a
        p = p * w_A / w_bar
        freqs.append(p)
    return np.array(freqs)

n_gen = 80
s_fd = 0.1

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Trajectories from different starting points
for p_start, color in [(0.05,'navy'),(0.20,'steelblue'),(0.50,'gray'),
                        (0.80,'coral'),(0.95,'firebrick')]:
    traj = freq_dep_selection_trajectory(p_start, s_fd, n_gen)
    ax1.plot(traj, color=color, lw=2, label=f'p₀={p_start}')
ax1.axhline(0.5, ls='--', color='black', lw=1, label='Equilibrium p*=0.5')
ax1.set_xlabel('Generation')
ax1.set_ylabel('Allele frequency')
ax1.set_title(f'Negative Frequency-Dependent Selection (s={s_fd})')
ax1.legend(fontsize=9)

# Compare with positive frequency dependence (rare allele disfavored: Allee effect)
def pos_freq_dep(p0, s_pos, n_gen):
    p = p0
    freqs = [p]
    for _ in range(n_gen):
        q = 1 - p
        w_A = 1 + s_pos * (p - 0.5)   # higher fitness when common
        w_a = 1 + s_pos * (q - 0.5)
        w_bar = p * w_A + q * w_a
        p = np.clip(p * w_A / w_bar, 0, 1)
        freqs.append(p)
    return np.array(freqs)

for p_start, color in [(0.45,'steelblue'),(0.55,'coral'),(0.49,'navy'),(0.51,'firebrick')]:
    traj = pos_freq_dep(p_start, s_fd, n_gen)
    ax2.plot(traj, color=color, lw=2, label=f'p₀={p_start}')
ax2.axhline(0.5, ls='--', color='black', lw=1, label='Unstable equilibrium')
ax2.set_xlabel('Generation')
ax2.set_title('Positive Frequency-Dependent Selection (Allee-like effect)')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Selection + drift: when does selection dominate?
# 2*Ne*s >> 1: selection effective; 2*Ne*s << 1: drift dominates

def wf_selection_drift(N, p0, s, h, n_gen, rng):
    """Wright-Fisher model with selection and genetic drift."""
    p = p0
    freqs = [p]
    for _ in range(n_gen):
        q = 1 - p
        w_bar = p**2 + 2*p*q*(1-h*s) + q**2*(1-s)
        p_sel = (p**2 + p*q*(1-h*s)) / w_bar   # after selection
        p = rng.binomial(2*N, p_sel) / (2*N)    # binomial drift
        freqs.append(p)
        if p == 0.0 or p == 1.0:
            freqs.extend([p] * (n_gen - len(freqs) + 1))
            break
    return np.array(freqs[:n_gen + 1])

N = 100
n_reps = 20
n_gen = 300

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, s_val, title in [
    (ax1, 0.05,  f'2Ns = {2*N*0.05:.0f} >> 1: selection dominates'),
    (ax2, 0.002, f'2Ns = {2*N*0.002:.1f} ≈ 1: drift and selection compete'),
]:
    n_fixed = 0
    for _ in range(n_reps):
        traj = wf_selection_drift(N, 0.05, s_val, h=0.5, n_gen=n_gen, rng=rng)
        ax.plot(traj, alpha=0.5, lw=0.9)
        if traj[-1] == 1.0:
            n_fixed += 1
    ax.set_title(title)
    ax.set_xlabel('Generation')
    ax.text(0.05, 0.92, f'{n_fixed}/{n_reps} fixed', transform=ax.transAxes, fontsize=10)

ax1.set_ylabel('Allele frequency')
plt.suptitle('Selection + Drift (N=100, p₀=0.05, h=0.5)', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Purifying selection and mutation-selection balance
# Deleterious recessive alleles are maintained by recurrent mutation even
# as selection removes them. Equilibrium: q_eq ≈ sqrt(mu/s)

def selection_trajectory_bad_allele(q0, s_del, h_val, n_gen):
    """Track frequency of a deleterious 'a' allele."""
    return 1 - selection_trajectory(1 - q0, s_del, h_val, n_gen)

# Purifying selection: dominance matters greatly
n_gen = 2000
q0_del = 0.4
s_del = 0.05

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

for h_val, ls, label in [
    (0.0, '-',  'Fully recessive (h=0): hidden in Aa'),
    (0.1, '--', 'Partially dominant (h=0.1)'),
    (0.5, ':',  'Additive (h=0.5): fastest removal'),
]:
    bad_traj = selection_trajectory_bad_allele(q0_del, s_del, h_val, n_gen)
    ax1.semilogy(bad_traj, ls=ls, lw=2, label=label)

ax1.set_xlabel('Generation')
ax1.set_ylabel('Freq of deleterious allele (log scale)')
ax1.set_title(f'Purifying Selection (s={s_del}, q₀={q0_del})')
ax1.legend(fontsize=9)

# Mutation-selection balance: q_eq = sqrt(mu/s) for fully recessive
mu_values = np.logspace(-7, -4, 50)
s_values_msb = [0.01, 0.05, 0.10]
for s_msb, color in zip(s_values_msb, ['navy', 'steelblue', 'coral']):
    q_eq = np.sqrt(mu_values / s_msb)   # fully recessive
    ax2.loglog(mu_values, q_eq, lw=2, color=color, label=f's={s_msb}')

ax2.set_xlabel('Mutation rate (mu)')
ax2.set_ylabel('Equilibrium freq of deleterious allele')
ax2.set_title('Mutation-Selection Balance: q_eq = sqrt(mu/s)')
ax2.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Summary: comparison of selection regimes
n_gen_summary = 300
p0_sum = 0.1
s_sum = 0.05

regimes = [
    ('Directional (h=0.5)', selection_trajectory(p0_sum, s_sum, 0.5, n_gen_summary), 'steelblue'),
    ('Dominant (h=0)',       selection_trajectory(p0_sum, s_sum, 0.0, n_gen_summary), 'seagreen'),
    ('Recessive (h=1)',      selection_trajectory(p0_sum, s_sum, 1.0, n_gen_summary), 'coral'),
    ('Neutral (s=0)',        selection_trajectory(p0_sum, 0.00, 0.5, n_gen_summary), 'gray'),
]

fig, ax = plt.subplots(figsize=(10, 5))
for label, traj, color in regimes:
    ax.plot(traj, color=color, lw=2, label=label)
ax.axhline(p0_sum, ls=':', color='black', lw=1)
ax.set_xlabel('Generation')
ax.set_ylabel('Frequency of advantageous allele')
ax.set_title(f'Selection Regimes Compared (s={s_sum}, p₀={p0_sum})')
ax.legend()
plt.tight_layout()
plt.show()

print('Generations to reach p > 0.9:')
for label, traj, _ in regimes:
    over = np.where(traj > 0.9)[0]
    gen_90 = over[0] if len(over) > 0 else '>300'
    print(f'  {label:<25}: {gen_90}')


### The 2Ns Rule: When Does Selection Win?

The fundamental competition between selection and drift is captured by the **2N_e s** criterion:

| 2N_e s | Outcome |
|---|---|
| ≪ 1 | Effectively neutral: drift dominates, allele behaves as if s=0 |
| ≈ 1 | Transition zone: both drift and selection matter |
| ≫ 1 | Selection dominates: allele frequency change is nearly deterministic |

This explains why purifying selection is less efficient in small populations:
weakly deleterious mutations (small s) with 2N_e s < 1 accumulate by drift.
This **nearly neutral theory** (Ohta) predicts more slightly deleterious substitutions
in species with smaller N_e (e.g., mammals vs. *Drosophila*).


---

## 4. Molecular Clock and Divergence Time

**Neutral theory** (Kimura, 1968): most mutations at the molecular level are selectively neutral. The rate of neutral substitution equals the mutation rate regardless of population size:

$$k = \mu$$

*Proof sketch*: in a diploid population of size N, there are 2N new mutations per generation per site. Each has fixation probability 1/(2N) (neutral). So the rate of new fixations = 2N × μ × 1/(2N) = μ.

This gives a **molecular clock**: divergence accumulates linearly with time:
$$T = \frac{d}{2k}$$

The factor 2 accounts for independent substitutions along both lineages.

In [ ]:
# Jukes-Cantor (JC69) correction for multiple hits
# Raw p-distance underestimates true divergence because some sites
# experience back-mutations and multiple substitutions.
# JC69 (equal rates, equal base freqs):
# d_JC = -(3/4) * ln(1 - (4/3)*p)

def jukes_cantor_distance(p_distance):
    """JC69 corrected distance. p_distance = fraction of differing sites."""
    arg = 1.0 - (4.0/3.0) * p_distance
    if arg <= 0:
        return np.inf   # saturation: correction fails
    return -(3.0/4.0) * np.log(arg)

p_range = np.linspace(0, 0.74, 300)
d_jc = np.array([jukes_cantor_distance(p) for p in p_range])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(p_range, p_range, '--', color='gray', label='No correction (p-distance)')
ax.plot(p_range[d_jc < 5], d_jc[d_jc < 5], color='steelblue', lw=2, label='JC69 corrected')
ax.set_xlabel('Observed p-distance (fraction of differing sites)')
ax.set_ylabel('Estimated true distance (substitutions per site)')
ax.set_title('Jukes-Cantor Correction for Multiple Hits')
ax.legend()
ax.set_ylim(0, 3)
plt.tight_layout()
plt.show()
print("At high p-distances (> 0.5), the true distance far exceeds the raw p-distance.")
print("At p = 0.75 (random sequence similarity), the correction diverges to infinity.")

In [ ]:
# Divergence time estimation: T = d_JC / (2 * k)

comparisons = {
    'Human-Chimpanzee (mt DNA)': {'p_dist': 0.072, 'k': 2.2e-7, 'fossil_mya': 5.5},
    'Human-Gorilla (mt DNA)':    {'p_dist': 0.095, 'k': 2.2e-7, 'fossil_mya': 7.5},
    'Human-Orangutan (mt DNA)':  {'p_dist': 0.133, 'k': 2.2e-7, 'fossil_mya': 13.0},
    'Mouse-Rat (nuclear)':       {'p_dist': 0.025, 'k': 2.5e-9, 'fossil_mya': 12.0},
}

print(f"{'Comparison':<38} {'p-dist':>7} {'d_JC':>7} {'T_est (Mya)':>12} {'Fossil (Mya)':>13}")
print('-' * 82)
for name, vals in comparisons.items():
    p = vals['p_dist']
    k = vals['k']
    d = jukes_cantor_distance(p)
    T_est = d / (2 * k) / 1e6
    fossil = vals['fossil_mya']
    print(f"{name:<38} {p:>7.3f} {d:>7.4f} {T_est:>12.1f} {fossil:>13.1f}")
print()
print("Molecular clock estimates agree with fossil record within ~20-30%,")
print("typical for mtDNA with a calibrated substitution rate.")

In [ ]:
# Relative rate test: detecting rate variation across lineages
# For taxa A, B, outgroup C: d(A,C) - d(B,C) ≈ 0 under strict clock

d_human_macaque = jukes_cantor_distance(0.231)
d_chimp_macaque  = jukes_cantor_distance(0.225)
d_human_chimp    = jukes_cantor_distance(0.072)

rate_diff = d_human_macaque - d_chimp_macaque
print("Relative Rate Test: Human vs Chimp, Outgroup = Macaque")
print(f"  d(Human, Macaque):  {d_human_macaque:.4f}")
print(f"  d(Chimp, Macaque):  {d_chimp_macaque:.4f}")
print(f"  d(Human, Chimp):    {d_human_chimp:.4f}")
print(f"  Rate difference:    {rate_diff:.4f}")
print()
print("Small difference => approximate molecular clock holds.")
print()
print("Note: 'Relaxed clocks' (BEAST, MrBayes) allow rates to vary")
print("across branches using uncorrelated log-normal or autocorrelated models.")

---

## 5. dN/dS (Ka/Ks) Ratio

The ratio of non-synonymous substitution rate (dN) to synonymous substitution rate (dS) is the most widely used test for natural selection on protein-coding genes:

| dN/dS (omega) | Interpretation |
|---|---|
| < 1 | Purifying (negative) selection — most non-syn mutations are deleterious |
| = 1 | Neutral evolution — non-syn and syn substitutions occur at equal rates |
| > 1 | Positive (adaptive) selection — non-syn mutations driven to fixation |

### Nei-Gojobori Method

1. Count synonymous sites (S) and non-synonymous sites (N) in each codon
2. Count synonymous differences (Sd) and non-synonymous differences (Nd)
3. Compute pS = Sd/S, pN = Nd/N
4. Apply JC69 correction to get dS and dN
5. omega = dN/dS

### Rate Heterogeneity Across Sites

Real sequences do not evolve at a uniform rate across all sites. Sites under stronger
functional constraint evolve more slowly. The **gamma (+G) model** captures this by
assuming rates are drawn from a Gamma distribution with shape parameter α:

- α < 1: highly heterogeneous (many invariant + some fast sites)
- α = 1: exponential distribution of rates
- α > 10: approximately uniform rates

The JC+G distance is computed by integrating the JC correction over the Gamma rate distribution.
Most phylogenetics software (IQ-TREE, RAxML, BEAST) estimates α from the data by maximum likelihood.


In [ ]:
# Genetic code table and synonymous site counting
GENETIC_CODE = {
    'TTT':'Phe','TTC':'Phe','TTA':'Leu','TTG':'Leu',
    'CTT':'Leu','CTC':'Leu','CTA':'Leu','CTG':'Leu',
    'ATT':'Ile','ATC':'Ile','ATA':'Ile','ATG':'Met',
    'GTT':'Val','GTC':'Val','GTA':'Val','GTG':'Val',
    'TCT':'Ser','TCC':'Ser','TCA':'Ser','TCG':'Ser',
    'CCT':'Pro','CCC':'Pro','CCA':'Pro','CCG':'Pro',
    'ACT':'Thr','ACC':'Thr','ACA':'Thr','ACG':'Thr',
    'GCT':'Ala','GCC':'Ala','GCA':'Ala','GCG':'Ala',
    'TAT':'Tyr','TAC':'Tyr','TAA':'Stop','TAG':'Stop',
    'CAT':'His','CAC':'His','CAA':'Gln','CAG':'Gln',
    'AAT':'Asn','AAC':'Asn','AAA':'Lys','AAG':'Lys',
    'GAT':'Asp','GAC':'Asp','GAA':'Glu','GAG':'Glu',
    'TGT':'Cys','TGC':'Cys','TGA':'Stop','TGG':'Trp',
    'CGT':'Arg','CGC':'Arg','CGA':'Arg','CGG':'Arg',
    'AGT':'Ser','AGC':'Ser','AGA':'Arg','AGG':'Arg',
    'GGT':'Gly','GGC':'Gly','GGA':'Gly','GGG':'Gly',
}
BASES = ['A', 'T', 'C', 'G']

def count_syn_sites(codon):
    """Count synonymous sites in a codon (Nei-Gojobori averaging)."""
    if codon not in GENETIC_CODE or GENETIC_CODE[codon] == 'Stop':
        return 0.0, 3.0
    aa_orig = GENETIC_CODE[codon]
    syn_sites = 0.0
    for pos in range(3):
        syn_count = 0
        total_nonstop = 0
        for base in BASES:
            if base == codon[pos]:
                continue
            mutant = codon[:pos] + base + codon[pos+1:]
            if mutant not in GENETIC_CODE or GENETIC_CODE[mutant] == 'Stop':
                continue
            total_nonstop += 1
            if GENETIC_CODE[mutant] == aa_orig:
                syn_count += 1
        if total_nonstop > 0:
            syn_sites += syn_count / total_nonstop
    return syn_sites, 3.0 - syn_sites

# Codons with many synonyms (e.g., Leu=6) have more synonymous sites
print("Synonymous sites per codon:")
for codon in ['ATG', 'TGG', 'CTG', 'GAA', 'GCT', 'TTA']:
    s, n = count_syn_sites(codon)
    print(f"  {codon} ({GENETIC_CODE[codon]:>3}): {s:.2f} syn sites, {n:.2f} non-syn sites")

In [ ]:
def pairwise_dn_ds(seq1, seq2):
    """Compute dN and dS between two aligned coding sequences (Nei-Gojobori)."""
    assert len(seq1) == len(seq2) and len(seq1) % 3 == 0
    S_total = N_total = Sd = Nd = 0.0

    for i in range(0, len(seq1), 3):
        c1, c2 = seq1[i:i+3], seq2[i:i+3]
        if 'N' in c1 or 'N' in c2:
            continue
        if c1 not in GENETIC_CODE or c2 not in GENETIC_CODE:
            continue
        if GENETIC_CODE[c1] == 'Stop' or GENETIC_CODE[c2] == 'Stop':
            continue

        s1, n1 = count_syn_sites(c1)
        s2, n2 = count_syn_sites(c2)
        S_total += (s1 + s2) / 2
        N_total += (n1 + n2) / 2

        n_diffs = sum(c1[j] != c2[j] for j in range(3))
        if n_diffs == 0:
            continue
        aa1, aa2 = GENETIC_CODE[c1], GENETIC_CODE[c2]
        if n_diffs == 1:
            (Sd if aa1 == aa2 else Nd).__class__  # trick to avoid branch
            if aa1 == aa2:
                Sd += 1
            else:
                Nd += 1
        else:
            # Approximate: multiple changes per codon
            if aa1 == aa2:
                Sd += n_diffs
            else:
                Nd += n_diffs

    pS = Sd / S_total if S_total > 0 else 0.0
    pN = Nd / N_total if N_total > 0 else 0.0
    dS = jukes_cantor_distance(pS) if pS < 0.74 else np.inf
    dN = jukes_cantor_distance(pN) if pN < 0.74 else np.inf
    omega = dN / dS if dS > 0 else np.inf

    return {'S': S_total, 'N': N_total, 'Sd': Sd, 'Nd': Nd,
            'pS': pS, 'pN': pN, 'dS': dS, 'dN': dN, 'omega': omega}

# Example: two 20-codon sequences
gene_seq1 = 'ATGGAACTGTTTGCCAAGACTGTGCGTCAGAACGATTCCATGGCGTTACTGCTGGGAAAG'
gene_seq2 = 'ATGGAGCTGTTCGCCAAGACGGTGCGTCAGAATGATTCCATGGCATTGCTGCTGGGTAAA'
# Pad to same length if needed
min_len = min(len(gene_seq1), len(gene_seq2))
min_len = (min_len // 3) * 3

result = pairwise_dn_ds(gene_seq1[:min_len], gene_seq2[:min_len])
print("Nei-Gojobori dN/dS Analysis:")
for key in ['S','N','Sd','Nd','pS','pN','dS','dN','omega']:
    print(f"  {key}: {result[key]:.4f}")
if result['omega'] < 1:
    print("  Interpretation: PURIFYING SELECTION")
elif result['omega'] > 1:
    print("  Interpretation: POSITIVE SELECTION")
else:
    print("  Interpretation: NEUTRAL EVOLUTION")

In [ ]:
# Sliding window dN/dS along a simulated viral gene
# Conserved core (purifying selection) vs. variable loop (positive selection)

rng2 = np.random.default_rng(99)

def random_codon(rng):
    valid = [c for c in GENETIC_CODE if GENETIC_CODE[c] != 'Stop']
    return rng.choice(valid)

def mutate_codon_syn(codon, rng):
    aa = GENETIC_CODE.get(codon, 'Stop')
    syns = [c for c in GENETIC_CODE if GENETIC_CODE[c] == aa and c != codon]
    return rng.choice(syns) if syns else codon

def mutate_codon_nonsyn(codon, rng):
    nonsyns = [c for c in GENETIC_CODE
               if GENETIC_CODE[c] not in (GENETIC_CODE[codon], 'Stop') and c != codon]
    return rng.choice(nonsyns) if nonsyns else codon

n_codons = 200
codons_ref = [random_codon(rng2) for _ in range(n_codons)]
codons_qry = list(codons_ref)

for i in range(n_codons):
    r = rng2.random()
    if 50 <= i < 100:
        if r < 0.35:
            codons_qry[i] = mutate_codon_nonsyn(codons_qry[i], rng2)
    elif 150 <= i < 200:
        if r < 0.10:
            codons_qry[i] = mutate_codon_nonsyn(codons_qry[i], rng2)
        elif r < 0.20:
            codons_qry[i] = mutate_codon_syn(codons_qry[i], rng2)
    else:
        if r < 0.20:
            codons_qry[i] = mutate_codon_syn(codons_qry[i], rng2)
        elif r < 0.22:
            codons_qry[i] = mutate_codon_nonsyn(codons_qry[i], rng2)

seq_ref = ''.join(codons_ref)
seq_qry = ''.join(codons_qry)

window = 30
step   = 5
win_centers = []
omega_vals  = []
for start in range(0, n_codons - window + 1, step):
    end = start + window
    r = pairwise_dn_ds(''.join(codons_ref[start:end]), ''.join(codons_qry[start:end]))
    if r['dS'] > 0 and r['dS'] < 5:
        win_centers.append((start + end) / 2)
        omega_vals.append(min(r['omega'], 5))

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(win_centers, omega_vals, 'o-', color='steelblue', ms=4, lw=1.5)
ax.axhline(1.0, ls='--', color='black', lw=1, label='dN/dS = 1 (neutral)')
ax.axvspan(50, 100, alpha=0.12, color='red',    label='Positive selection region')
ax.axvspan(150, 200, alpha=0.08, color='orange', label='Moderate selection region')
ax.set_xlabel('Codon position (window center)')
ax.set_ylabel('dN/dS (omega)')
ax.set_title('Sliding Window dN/dS Along a Simulated Viral Gene')
ax.legend()
plt.tight_layout()
plt.show()

### Caveats of dN/dS Analysis

1. **Saturation**: at high divergence (dS > 1-2), multiple hits make dS unreliable. The JC69 correction diverges at p = 0.75.
2. **Gene-averaging**: if only a few codons are positively selected, the gene-average omega can still be < 1. Site-specific models (PAML M2a, M8) are more sensitive.
3. **Alignment errors**: frameshifts or misaligned regions produce spurious stop codons and inflated dN.
4. **Codon usage bias**: CpG mutation bias and selection on synonymous codon use can inflate dS independently.
5. **Short sequences**: few synonymous sites = high variance in dS estimate.

---

## 6. Neutrality Tests

Neutrality tests detect departures from what we'd expect under neutral evolution in a randomly mating population of constant size.

### Tajima's D

Two estimators of $\theta = 4N_e\mu$:
- **pi (θ_π)**: average pairwise nucleotide differences
- **Watterson (θ_W)**: based on the number of segregating sites S

$$D = \frac{\hat\pi - \hat\theta_W}{\sqrt{\text{Var}(\hat\pi - \hat\theta_W)}}$$

Under strict neutrality in a stable population: D ≈ 0.
- **D < 0**: excess rare variants — recent sweep or population expansion
- **D > 0**: excess intermediate-frequency variants — balancing selection or bottleneck

In [ ]:
def tajimas_d(seqs):
    """Compute Tajima's D for a list of aligned DNA strings."""
    n = len(seqs)
    L = len(seqs[0])
    assert all(len(s) == L for s in seqs)

    S = sum(1 for j in range(L) if len(set(s[j] for s in seqs)) > 1)
    if S == 0:
        return 0.0, 0.0, 0.0, 0

    # Average pairwise differences (pi estimator)
    total_diffs = sum(
        sum(seqs[i][k] != seqs[j][k] for k in range(L))
        for i in range(n) for j in range(i+1, n)
    )
    n_pairs = n * (n - 1) // 2
    pi = total_diffs / n_pairs

    # Watterson estimator
    a1 = sum(1/k for k in range(1, n))
    a2 = sum(1/k**2 for k in range(1, n))
    theta_W = S / a1

    # Variance (Tajima 1989)
    b1 = (n + 1) / (3 * (n - 1))
    b2 = 2 * (n**2 + n + 3) / (9 * n * (n - 1))
    c1 = b1 - 1/a1
    c2 = b2 - (n + 2)/(a1 * n) + a2/a1**2
    e1 = c1 / a1
    e2 = c2 / (a1**2 + a2)
    var_d = e1 * S + e2 * S * (S - 1)

    if var_d <= 0:
        return pi, theta_W, 0.0, S
    return pi, theta_W, (pi - theta_W) / np.sqrt(var_d), S

def generate_sequences(n_seq, L, n_seg_sites, rng, freq_spectrum='uniform'):
    """Generate synthetic alignments with controlled allele frequency spectra."""
    ref  = ''.join(rng.choice(list('ATCG')) for _ in range(L))
    seqs = [list(ref) for _ in range(n_seq)]
    for pos in rng.choice(L, size=n_seg_sites, replace=False):
        alt = rng.choice([b for b in 'ATCG' if b != ref[pos]])
        if freq_spectrum == 'rare':
            seqs[rng.integers(n_seq)][pos] = alt
        elif freq_spectrum == 'common':
            for c in rng.choice(n_seq, size=rng.integers(n_seq//3, 2*n_seq//3), replace=False):
                seqs[c][pos] = alt
        else:
            for c in rng.choice(n_seq, size=rng.integers(1, n_seq), replace=False):
                seqs[c][pos] = alt
    return [''.join(s) for s in seqs]

rng3 = np.random.default_rng(7)
print(f"{'Scenario':<45} {'S':>4} {'pi':>6} {'theta_W':>8} {'D':>8}")
print('-' * 76)
for scenario, spectrum in [
    ('Neutral (uniform freq spectrum)', 'uniform'),
    ('Expansion / selective sweep (rare variants)', 'rare'),
    ('Balancing selection / bottleneck (common vars)', 'common'),
]:
    seqs = generate_sequences(20, 500, 40, rng3, freq_spectrum=spectrum)
    pi, tW, D, S = tajimas_d(seqs)
    print(f"{scenario:<45} {S:>4} {pi:>6.3f} {tW:>8.3f} {D:>8.3f}")

In [ ]:
# Compute Tajima's D for the three scenarios and print summary
rng3_td = np.random.default_rng(701)

print(f"{'Scenario':<50} {'S':>4} {'pi':>6} {'theta_W':>8} {'D':>8}")
print('-' * 80)
for scenario, spectrum in [
    ('Neutral (uniform freq spectrum)', 'uniform'),
    ('Population expansion / selective sweep (rare vars)', 'rare'),
    ('Balancing selection / bottleneck (common vars)', 'common'),
]:
    seqs = generate_sequences(20, 500, 40, rng3_td, freq_spectrum=spectrum)
    pi, tW, D, S = tajimas_d(seqs)
    print(f"{scenario:<50} {S:>4} {pi:>6.3f} {tW:>8.3f} {D:>8.3f}")


In [ ]:
# Visualise the allele frequency spectrum (AFS) for each scenario
# The AFS directly shows why Tajima's D differs

rng3b = np.random.default_rng(77)
n_seq = 20
L = 500
n_seg = 40

scenarios = {
    'Neutral':          generate_sequences(n_seq, L, n_seg, rng3b, 'uniform'),
    'Expansion/Sweep':  generate_sequences(n_seq, L, n_seg, rng3b, 'rare'),
    'Balancing/Bottleneck': generate_sequences(n_seq, L, n_seg, rng3b, 'common'),
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
for ax, (name, seqs) in zip(axes, scenarios.items()):
    # Compute derived allele frequency for each segregating site
    afs = []
    for j in range(L):
        site = [s[j] for s in seqs]
        alleles = set(site)
        if len(alleles) > 1:
            counts = {a: site.count(a) for a in alleles}
            minor_count = min(counts.values())
            afs.append(minor_count / n_seq)
    if afs:
        ax.hist(afs, bins=np.linspace(0, 1, 11), color='steelblue', edgecolor='white')
    pi, tW, D, S = tajimas_d(seqs)
    ax.set_title(f'{name}\nTajima\'s D = {D:.2f}')
    ax.set_xlabel('Minor allele frequency')
    ax.set_ylabel('Count of SNPs')

plt.suptitle('Allele Frequency Spectra and Tajima\'s D', y=1.02)
plt.tight_layout()
plt.show()

### Site Frequency Spectrum (SFS)

The **site frequency spectrum** (SFS) plots the distribution of variant allele frequencies
across all segregating sites. Its shape directly determines Tajima's D:

- **Neutral SFS** (standard coalescent): L-shaped — many rare variants, few common ones
- **Sweep / expansion SFS**: even more L-shaped — excess singletons (D < 0)
- **Balancing selection SFS**: flat or hill-shaped — many intermediate-frequency variants (D > 0)

The expected neutral SFS for a sample of n sequences is $E[\xi_k] = \theta / k$
where $\xi_k$ is the count of sites where the derived allele appears in k of n sequences.


In [ ]:
# McDonald-Kreitman test
#             Synonymous    Non-synonymous
#  Polymorphic    Ps              Pn
#  Divergent      Ds              Dn
#
# NI = (Pn/Ps)/(Dn/Ds); alpha = 1 - (Ds*Pn)/(Dn*Ps)

# Real data: Drosophila Adh gene (McDonald & Kreitman 1991, Nature)
Ps = 43   # synonymous polymorphisms (within D. melanogaster)
Pn =  2   # non-synonymous polymorphisms
Ds = 17   # synonymous fixed differences (vs D. simulans)
Dn =  7   # non-synonymous fixed differences

NI    = (Pn / Ps) / (Dn / Ds)
alpha = 1 - (Ds * Pn) / (Dn * Ps)

table = np.array([[Pn, Ps], [Dn, Ds]])
odds_ratio, p_fisher = stats.fisher_exact(table)

print("McDonald-Kreitman Test: Drosophila Adh gene")
print(f"{'':25} {'Synonymous':>12} {'Non-synonymous':>16}")
print(f"{'Polymorphic':25} {Ps:>12} {Pn:>16}")
print(f"{'Divergent':25} {Ds:>12} {Dn:>16}")
print()
print(f"Neutrality Index (NI):  {NI:.3f}   (< 1 => positive selection)")
print(f"Alpha (adaptive frac):  {alpha:.3f}   ({alpha*100:.1f}% of Dn is adaptive)")
print(f"Fisher's exact p-value: {p_fisher:.4f}")
if p_fisher < 0.05 and alpha > 0:
    print()
    print("Conclusion: Significant excess of fixed non-syn substitutions.")
    print("Strong evidence for positive selection on Adh.")

### Confounding by Demography

Both Tajima's D and the MK test are sensitive to demographic history:

| Demographic event | Effect on Tajima's D | Effect on MK |
|---|---|---|
| Population expansion | D < 0 (mimics sweep) | Inflates Pn/Ps (NI > 1 spuriously) |
| Bottleneck | D > 0 (mimics balancing selection) | Can mask positive selection |
| Population structure | Elevated D | Less affected if done within populations |

**Corrective approaches:** compare across many loci (genome-wide), use coalescent demographic null models, or apply the Direction of Selection (DoS) statistic.

### Extensions of the MK Framework

The McDonald-Kreitman test has been extended in several important ways:

1. **Genome-wide MK**: average alpha over thousands of genes gives a robust
   estimate of the proportion of adaptive evolution in a genome (typically 10-50%
   in *Drosophila*, lower in humans due to smaller Ne and stronger drift).

2. **Direction of Selection (DoS)**: DoS = Dn/(Dn+Ds) − Pn/(Pn+Ps). DoS > 0
   indicates positive selection; DoS < 0 indicates slightly deleterious mutations
   contribute to polymorphism.

3. **Asymptotic MK**: removes the confounding effect of slightly deleterious
   polymorphisms by fitting alpha as a function of derived allele frequency and
   extrapolating to frequency = 1.

4. **Site-specific MK**: applied to individual codons or regulatory elements
   rather than whole genes.


---

## 7. Population Structure

Geographic, ecological, or cultural barriers create **population structure** — systematic differences in allele frequencies between groups.

### Fst: Fixation Index

$$F_{ST} = \frac{H_T - H_S}{H_T}$$

where $H_T$ = expected heterozygosity in the pooled population, $H_S$ = average expected heterozygosity within subpopulations.

**Wright's qualitative scale:** 0–0.05 (little), 0.05–0.15 (moderate), 0.15–0.25 (great), > 0.25 (very great differentiation)

In [ ]:
def compute_fst(pop_allele_freqs):
    """Fst for a biallelic locus across populations (equal sample sizes)."""
    freqs = np.array(pop_allele_freqs)
    H_S = np.mean(2 * freqs * (1 - freqs))     # within-pop heterozygosity
    p_bar = np.mean(freqs)
    H_T = 2 * p_bar * (1 - p_bar)              # total heterozygosity
    if H_T == 0:
        return 0.0, H_S, H_T
    return (H_T - H_S) / H_T, H_S, H_T

# Pigmentation and other SNPs across three human populations (approx 1000G)
snp_data = pd.DataFrame({
    'SNP':   ['rs1426654','rs16891982','rs12913832','rs4988235','rs6152'],
    'Gene':  ['SLC24A5',  'SLC45A2',   'HERC2/OCA2','LCT',     'AR'],
    'Trait': ['Skin pigmentation','Skin pigmentation','Eye color',
              'Lactase persistence','Androgen receptor'],
    'p_AFR': [0.02, 0.01, 0.05, 0.14, 0.55],
    'p_EUR': [0.97, 0.82, 0.73, 0.72, 0.42],
    'p_EAS': [0.99, 0.93, 0.20, 0.16, 0.35],
})
snp_data['Fst'] = [
    compute_fst([r['p_AFR'], r['p_EUR'], r['p_EAS']])[0]
    for _, r in snp_data.iterrows()
]
print(snp_data[['SNP','Gene','Trait','p_AFR','p_EUR','p_EAS','Fst']].to_string(index=False))
print()
print("Pigmentation SNPs (SLC24A5, SLC45A2) show Fst >> 0.25: very great")
print("differentiation driven by adaptation to UV radiation levels.")

In [ ]:
# Fst and gene flow: Wright's island model
# Fst ≈ 1 / (1 + 4*Ne*m) where m = migration rate per generation
# High gene flow => low Fst; isolated populations => high Fst

migration_rates = np.logspace(-4, -1, 100)   # m from 0.0001 to 0.1
Ne_island = 1000
fst_theory = 1 / (1 + 4 * Ne_island * migration_rates)

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogx(migration_rates, fst_theory, color='steelblue', lw=2)
ax.axhline(0.25, ls='--', color='red',    lw=1, label='Very great (Fst=0.25)')
ax.axhline(0.15, ls='--', color='orange', lw=1, label='Great (Fst=0.15)')
ax.axhline(0.05, ls='--', color='green',  lw=1, label='Little (Fst=0.05)')
ax.set_xlabel('Migration rate (m) per generation')
ax.set_ylabel('Fst')
ax.set_title(f"Wright's Island Model: Fst = 1/(1+4N_e m), N_e={Ne_island}")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print("Fst at key migration rates:")
for m in [0.001, 0.005, 0.01, 0.05]:
    fst_m = 1 / (1 + 4 * Ne_island * m)
    print(f"  m={m:.3f}: Fst = {fst_m:.3f}")


In [ ]:
# Linkage disequilibrium (LD): r-squared between SNP pairs
# D = f(AB) - f(A)*f(B);
# r^2 = D^2 / (p_A * q_a * p_B * q_b)

def compute_ld_matrix(haplotypes):
    """r^2 matrix for all SNP pairs.
    haplotypes: (n_haplotypes x n_snps) array of 0/1.
    """
    n_hap, n_sites = haplotypes.shape
    freqs = haplotypes.mean(axis=0)
    r2_matrix = np.eye(n_sites)
    for i, j in combinations(range(n_sites), 2):
        p_A, p_B = freqs[i], freqs[j]
        D = np.mean(haplotypes[:, i] * haplotypes[:, j]) - p_A * p_B
        denom = p_A * (1 - p_A) * p_B * (1 - p_B)
        r2 = D**2 / denom if denom > 0 else 0.0
        r2_matrix[i, j] = r2_matrix[j, i] = r2
    return r2_matrix

rng4 = np.random.default_rng(13)
n_hap = 200
n_snps = 20
positions_kb  = np.linspace(0, 50, n_snps)
recomb_rate   = 1e-8
n_generations = 100

founder_A = rng4.integers(0, 2, size=n_snps)
founder_B = 1 - founder_A
haplotypes = np.zeros((n_hap, n_snps), dtype=int)
for h in range(n_hap):
    current = founder_A.copy() if rng4.random() < 0.5 else founder_B.copy()
    for snp_idx in range(1, n_snps):
        dist_bp = (positions_kb[snp_idx] - positions_kb[snp_idx-1]) * 1000
        if rng4.random() < 1 - np.exp(-recomb_rate * dist_bp * n_generations):
            other = founder_B if (current[0] == founder_A[0]) else founder_A
            current[snp_idx:] = other[snp_idx:]
    haplotypes[h] = current

r2_matrix = compute_ld_matrix(haplotypes)
pair_dist = [abs(positions_kb[j] - positions_kb[i])
             for i, j in combinations(range(n_snps), 2)]
pair_r2   = [r2_matrix[i, j]
             for i, j in combinations(range(n_snps), 2)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
im = ax1.imshow(r2_matrix, cmap='YlOrRd', vmin=0, vmax=1, aspect='auto')
ax1.set_title('Linkage Disequilibrium (r²) Matrix')
ax1.set_xlabel('SNP index')
ax1.set_ylabel('SNP index')
plt.colorbar(im, ax=ax1, label='r²')

ax2.scatter(pair_dist, pair_r2, alpha=0.4, s=15, color='steelblue')
dist_range = np.linspace(0, 50, 200)
expected_r2 = 1.0 / (1.0 + 4 * 10_000 * recomb_rate * 1000 * dist_range)
ax2.plot(dist_range, expected_r2, 'r-', lw=2, label='Expected: 1/(1+4N_e r d)')
ax2.set_xlabel('Physical distance (kb)')
ax2.set_ylabel('r²')
ax2.set_title('LD Decay with Physical Distance')
ax2.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Genome-wide Fst distribution under neutrality
# Most neutral loci have low Fst; outlier loci may signal local adaptation
rng_fst = np.random.default_rng(2024)
n_loci = 500
n_populations = 3

# Simulate allele frequencies from Dirichlet-like process with moderate structure
# Using a simple model: p_ancestral ~ Uniform, then each pop drifts independently
Ne_drift = 1000
n_gen_drift = 200
p_ancestral = rng_fst.uniform(0.1, 0.9, size=n_loci)

fst_loci = []
for i in range(n_loci):
    # Each population independently drifts from the ancestral frequency
    pop_freqs = []
    for _ in range(n_populations):
        p = p_ancestral[i]
        for _ in range(n_gen_drift):
            p = rng_fst.binomial(2 * Ne_drift, p) / (2 * Ne_drift)
        pop_freqs.append(p)
    fst, _, _ = compute_fst(pop_freqs)
    fst_loci.append(fst)

fst_loci = np.array(fst_loci)
print(f"Genome-wide Fst summary ({n_loci} neutral loci, Ne={Ne_drift}, {n_gen_drift} gen):")
print(f"  Mean Fst:   {fst_loci.mean():.4f}")
print(f"  Median Fst: {np.median(fst_loci):.4f}")
print(f"  95th pct:   {np.percentile(fst_loci, 95):.4f}  (outlier threshold)")
print(f"  Max Fst:    {fst_loci.max():.4f}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(fst_loci, bins=40, color='steelblue', edgecolor='white')
ax.axvline(np.percentile(fst_loci, 95), ls='--', color='red', lw=1.5,
           label=f'95th percentile = {np.percentile(fst_loci,95):.3f}')
ax.set_xlabel('Fst')
ax.set_ylabel('Number of loci')
ax.set_title(f'Genome-wide Fst Distribution ({n_loci} neutral loci)')
ax.legend()
plt.tight_layout()
plt.show()
print("\nLoci above the 95th Fst percentile are candidate targets of local adaptation.")


---

## 8. Exercises

These exercises reinforce the core concepts. Each is self-contained using the functions defined above.

### Exercise 1: Drift and Fixation Probability vs Initial Frequency

Under neutral drift, the probability that an allele starting at frequency *p₀* eventually fixes is exactly *p₀* (Kimura 1962). Verify this by simulation and plot the observed fixation probability against the theoretical expectation across multiple starting frequencies.

In [ ]:
rng_ex1 = np.random.default_rng(1001)
N_ex1   = 50
n_reps_ex1 = 300
max_gen_ex1 = 5000
p0_values = np.linspace(0.05, 0.95, 15)

fix_probs_obs = []
for p0_test in p0_values:
    n_fixed = 0
    for _ in range(n_reps_ex1):
        p = p0_test
        for _ in range(max_gen_ex1):
            p = rng_ex1.binomial(2 * N_ex1, p) / (2 * N_ex1)
            if p == 0.0 or p == 1.0:
                break
        if p == 1.0:
            n_fixed += 1
    fix_probs_obs.append(n_fixed / n_reps_ex1)

errors = np.abs(np.array(fix_probs_obs) - p0_values)
print(f"Mean absolute error from theoretical P(fix) = p₀: {errors.mean():.3f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot([0, 1], [0, 1], '--', color='black', label='Theory: P(fix) = p₀')
ax.scatter(p0_values, fix_probs_obs, color='steelblue', s=60, zorder=5, label='Simulated')
ax.set_xlabel('Initial allele frequency (p₀)')
ax.set_ylabel('Probability of fixation')
ax.set_title(f'Exercise 1: Fixation Probability vs p₀ (N={N_ex1}, {n_reps_ex1} reps)')
ax.legend()
plt.tight_layout()
plt.show()

### Exercise 2: Compute dN/dS for Two Aligned Coding Sequences

Given two aligned coding sequences from two species, use the Nei-Gojobori method to compute dN, dS, and omega. Interpret the result in terms of selection.

In [ ]:
# Exercise 2: SARS-CoV-2 Spike RBD vs a bat coronavirus RaTG13-like ortholog
# 90 nt excerpt from the receptor-binding domain

spike_human = (
    'ATG TAT CAG TTT AAT GGT GTG GAT CGT TAT ATG '
    'GCA GTG CGT AAT GGT CAC AAT GAA GGT AAA CTG '
    'GTG ACT CGT GAT GCT ATT ATG CAG ACT GGA GGA'
).replace(' ', '')

spike_bat = (
    'ATG TAC CAA TTT AAC GGT GTA GAT CGC TAT ATG '
    'GCT GTC CGC AAT GGT CAT AAT GAA GGA AAA CTT '
    'GTT ACA AGA GAT GCC ATC ATG CAA ACC GGA GGT'
).replace(' ', '')

assert len(spike_human) == len(spike_bat) == 90
result_ex2 = pairwise_dn_ds(spike_human, spike_bat)

print("Exercise 2: Spike RBD -- Human SARS-CoV-2 vs Bat RaTG13-like")
print(f"  dS = {result_ex2['dS']:.4f}   dN = {result_ex2['dN']:.4f}   omega = {result_ex2['omega']:.4f}")
print()
omega_ex2 = result_ex2['omega']
if omega_ex2 < 0.5:
    print("Strong purifying selection: most amino acid changes are deleterious.")
    print("Consistent with structural constraint in the receptor-binding domain.")
elif omega_ex2 < 1.0:
    print("Moderate purifying selection.")
elif omega_ex2 > 1.5:
    print("Positive/adaptive selection signal (omega > 1) -- host-switching?")
else:
    print("Approximately neutral evolution.")

### Exercise 3: Calculate Tajima's D for a Set of Sequences

Given a multiple sequence alignment, compute Tajima's D and interpret whether the data is consistent with neutral evolution, a recent population expansion, or balancing selection.

In [ ]:
# Exercise 3: G6PD-inspired alignment
# G6PD is under balancing selection in malaria-endemic regions:
# deficiency alleles maintained at intermediate frequency because
# heterozygous females have partial malaria resistance.

rng_ex3 = np.random.default_rng(303)

seqs_g6pd    = generate_sequences(15, 300, 35, rng_ex3, freq_spectrum='common')
seqs_neutral = generate_sequences(15, 300, 35, rng_ex3, freq_spectrum='uniform')
seqs_sweep   = generate_sequences(15, 300, 35, rng_ex3, freq_spectrum='rare')

pi_g, tW_g, D_g, S_g = tajimas_d(seqs_g6pd)
pi_n, tW_n, D_n, S_n = tajimas_d(seqs_neutral)
pi_s, tW_s, D_s, S_s = tajimas_d(seqs_sweep)

print("Exercise 3: Tajima's D under three scenarios (15 seqs, 300 bp)")
print(f"{'Scenario':<48} {'S':>4} {'pi':>6} {'theta_W':>8} {'D':>8}")
print('-' * 80)
print(f"{'G6PD-like (balancing selection)':48} {S_g:>4} {pi_g:>6.3f} {tW_g:>8.3f} {D_g:>8.3f}")
print(f"{'Neutral control':48} {S_n:>4} {pi_n:>6.3f} {tW_n:>8.3f} {D_n:>8.3f}")
print(f"{'Selective sweep / expansion':48} {S_s:>4} {pi_s:>6.3f} {tW_s:>8.3f} {D_s:>8.3f}")
print()
print(f"G6PD-like D = {D_g:.3f}: excess intermediate-freq variants => balancing selection")
print(f"Sweep D     = {D_s:.3f}: excess rare variants => sweep or expansion")

### Exercise 4: Compute Pairwise Fst Between Populations

Given allele frequency data for multiple SNPs across three populations, compute pairwise Fst for each SNP and each population pair. Identify the most differentiated SNPs and interpret them biologically.

In [ ]:
ex4_data = pd.DataFrame({
    'SNP':   ['rs1426654','rs16891982','rs12913832','rs4988235','rs6152',
               'rs1800562','rs1805007', 'rs4680',    'rs53576',  'rs1042522',
               'rs9939609','rs4994'],
    'Gene':  ['SLC24A5',  'SLC45A2',   'HERC2',     'LCT',      'AR',
               'HFE',      'MC1R',      'COMT',      'OXTR',     'TP53',
               'FTO',      'ADRB3'],
    'p_AFR': [0.02, 0.01, 0.05, 0.14, 0.55, 0.03, 0.02, 0.45, 0.38, 0.79, 0.46, 0.10],
    'p_EUR': [0.97, 0.82, 0.73, 0.72, 0.42, 0.07, 0.12, 0.52, 0.42, 0.60, 0.52, 0.23],
    'p_EAS': [0.99, 0.93, 0.20, 0.16, 0.35, 0.01, 0.01, 0.72, 0.39, 0.62, 0.42, 0.15],
})

pop_pairs = [('AFR','EUR'), ('AFR','EAS'), ('EUR','EAS')]
for p1, p2 in pop_pairs:
    ex4_data[f'Fst_{p1}_{p2}'] = [
        compute_fst([row[f'p_{p1}'], row[f'p_{p2}']])[0]
        for _, row in ex4_data.iterrows()
    ]

print(ex4_data[['SNP','Gene','Fst_AFR_EUR','Fst_AFR_EAS','Fst_EUR_EAS']]
      .to_string(index=False, float_format='{:.3f}'.format))
print()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (p1, p2) in zip(axes, pop_pairs):
    col = f'Fst_{p1}_{p2}'
    ax.barh(ex4_data['Gene'], ex4_data[col], color='steelblue')
    ax.axvline(0.15, ls='--', color='orange', lw=1.2, label='0.15 (great diff.)')
    ax.axvline(0.25, ls='--', color='red',    lw=1.2, label='0.25 (very great)')
    ax.set_xlabel('Fst')
    ax.set_title(f'{p1} vs {p2}')
    ax.set_xlim(0, 1)
    ax.legend(fontsize=8)

plt.suptitle('Exercise 4: Pairwise Fst by SNP and Population Pair', y=1.01)
plt.tight_layout()
plt.show()

ex4_data['max_Fst'] = ex4_data[['Fst_AFR_EUR','Fst_AFR_EAS','Fst_EUR_EAS']].max(axis=1)
top_snp = ex4_data.loc[ex4_data['max_Fst'].idxmax()]
print(f"Most differentiated SNP: {top_snp['SNP']} in {top_snp['Gene']} (max Fst={top_snp['max_Fst']:.3f})")

---

## Summary

| Topic | Key concept | Key metric / method |
|---|---|---|
| Hardy-Weinberg equilibrium | Null model for an ideal population | p² + 2pq + q² = 1; chi-squared test |
| Inbreeding coefficient | Fractional reduction in heterozygosity | F = 1 − H_obs/H_exp |
| Genetic drift | Random frequency change in finite populations | Wright-Fisher binomial; Ne |
| Bottleneck / Ne | Transient size reduction dominates long-run drift | Harmonic mean Ne |
| Directional selection | Advantageous allele sweeps to fixation | Coefficient s; 2Ns rule |
| Balancing selection | Stable polymorphism via heterozygote advantage | Overdominance; sickle-cell example |
| Purifying selection | Deleterious alleles removed (slowly if recessive) | Mutation-selection balance: q_eq = sqrt(mu/s) |
| Freq-dependent selection | Fitness depends on allele frequency | Negative: stabilizing; Positive: bistable |
| Molecular clock | Neutral substitutions accumulate at rate mu | T = d/(2k); Jukes-Cantor correction |
| Relative rate test | Detects rate variation across lineages | d(A,C) − d(B,C) ≈ 0 under strict clock |
| dN/dS (omega) | Non-syn vs syn substitution rate ratio | Nei-Gojobori method; omega < 1, = 1, > 1 |
| Tajima's D | Compares pi to theta_W | D < 0 (sweep/expansion), D > 0 (bottleneck/balancing) |
| McDonald-Kreitman | Polymorphism vs divergence, syn vs non-syn | NI and alpha (adaptive fraction) |
| Fst | Between-population differentiation | Fst = (H_T − H_S) / H_T |
| Linkage disequilibrium | Non-random allele associations; recombination erodes LD | r² = D² / (p_A q_a p_B q_b) |

---[< Previous: Population Genetics + Molecular Evolution: Over...](01_population_genetics_and_molecular_evolution.ipynb) | [Tier 3: Applied Bioinformatics Overview](../README.md) | [Next: Numerical Methods for Bioinformatics >](../16_Numerical_Methods_for_Bioinformatics/01_numerical_methods_for_bioinformatics.ipynb)---